# Translate SRT with Gemini

Source package: https://github.com/MaKTaiL/gemini-srt-translator

Edit only the config cell, then run all cells. The notebook can either upload `.srt` files directly and auto-download translated output, or read `.srt` files from a Google Drive folder and save translated files to another Drive folder.


In [ ]:
# ===============================================================
# CONFIG - EDIT ONLY THIS CELL, THEN RUNTIME > RUN ALL
# ===============================================================
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

# Choose input mode:
# - "upload": choose SRT file(s) directly; outputs auto-download in Colab.
# - "drive": read SRT file(s) from DRIVE_INPUT_DIR; outputs are saved to DRIVE_OUTPUT_DIR.
INPUT_MODE = "upload"
AUTO_DOWNLOAD = True

# Optional explicit file for upload/local mode. Leave None to open the Colab file chooser.
INPUT_FILE = None

# Used for upload mode. In Colab this should normally stay /content.
DEFAULT_OUTPUT_DIR = Path("/content" if IN_COLAB else "./translated_srt")

# Local Jupyter input folder used outside Colab when INPUT_MODE="upload" cannot open a Colab chooser.
LOCAL_INPUT_DIR = Path("./input_srt")

# Google Drive folder mode. Change these to your real Drive paths, then use INPUT_MODE="drive".
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/input_srt")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/output_srt")
RECURSIVE_DRIVE_SEARCH = True
PRESERVE_DRIVE_SUBFOLDERS = True

# Translation job settings.
TARGET_LANGUAGE = "Vietnamese"
OUTPUT_SUFFIX = None  # None => auto-generate from TARGET_LANGUAGE, for example input.vi.srt

# Model selection. Set MODEL_NAME to None in code if you want the translator package default.
MODEL_NAME = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro", "custom"]
CUSTOM_MODEL_NAME = ""  # Used only when MODEL_NAME="custom", for example "gemini-2.5-flash".

# Output policy. True keeps previous output safe by translating to a temp file first, then replacing it after success.
# False fails if the target output already exists.
OVERWRITE_OUTPUT = True

# Optional package settings.
BATCH_SIZE = None
START_LINE = None
DESCRIPTION = None
FREE_QUOTA = True
SKIP_UPGRADE_CHECK = True
QUIET = False

SUPPORTED_INPUT_SUFFIXES = {".srt"}


In [ ]:
# Workaround for occasional Colab/pip locale issues.
import locale
locale.getpreferredencoding = lambda: "UTF-8"

%pip install -q --upgrade gemini-srt-translator

In [ ]:
import os
from pathlib import Path

import gemini_srt_translator as gst

try:
    from google.colab import drive, files, userdata
except Exception:
    drive = None
    files = None
    userdata = None


def get_secret(secret_name: str, env_name: str | None = None) -> str | None:
    env_name = env_name or secret_name
    value = os.environ.get(env_name)
    if value:
        return value.strip()
    if userdata is not None:
        try:
            value = userdata.get(secret_name)
            return value.strip() if value else None
        except Exception:
            return None
    return None


def iter_input_files(input_dir: Path, suffixes: set[str], recursive: bool = True) -> list[Path]:
    pattern = "**/*" if recursive else "*"
    return [
        path for path in sorted(input_dir.glob(pattern))
        if path.is_file() and path.suffix.lower() in suffixes
    ]


def upload_srt_files() -> list[Path]:
    if files is None:
        raise RuntimeError(
            "Direct file upload is only available in Google Colab. "
            "Set INPUT_FILE to a local .srt path, or put files in LOCAL_INPUT_DIR."
        )

    print("Choose SRT file(s) from your computer.")
    uploaded = files.upload()
    selected_files = []
    for filename in uploaded.keys():
        file_path = Path("/content") / filename
        if file_path.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
            raise ValueError(f"Uploaded file is not an SRT: {filename}")
        selected_files.append(file_path)
    if not selected_files:
        raise ValueError("No .srt file was uploaded.")
    return selected_files


def collect_input_files() -> tuple[list[Path], Path, Path | None]:
    if INPUT_MODE == "upload":
        output_root = DEFAULT_OUTPUT_DIR
        output_root.mkdir(parents=True, exist_ok=True)

        if INPUT_FILE:
            input_path = Path(INPUT_FILE)
            if not input_path.exists():
                raise FileNotFoundError(f"Input file not found: {input_path}")
            if input_path.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
                raise ValueError(f"Input file is not an SRT: {input_path}")
            return [input_path], output_root, input_path.parent

        if IN_COLAB:
            return upload_srt_files(), output_root, None

        if not LOCAL_INPUT_DIR.exists():
            raise FileNotFoundError(
                "Not running in Colab. Put .srt files in LOCAL_INPUT_DIR, or run this notebook in Google Colab for direct upload."
            )
        input_files = iter_input_files(LOCAL_INPUT_DIR, SUPPORTED_INPUT_SUFFIXES, RECURSIVE_DRIVE_SEARCH)
        if not input_files:
            raise FileNotFoundError(f"No .srt files found in: {LOCAL_INPUT_DIR}")
        return input_files, output_root, LOCAL_INPUT_DIR

    if INPUT_MODE == "drive":
        if not IN_COLAB or drive is None:
            raise RuntimeError("Google Drive folder mode is available in Google Colab only.")
        drive.mount("/content/drive")
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        input_files = iter_input_files(DRIVE_INPUT_DIR, SUPPORTED_INPUT_SUFFIXES, RECURSIVE_DRIVE_SEARCH)
        if not input_files:
            raise FileNotFoundError(f"No .srt files found in: {DRIVE_INPUT_DIR}")
        return input_files, DRIVE_OUTPUT_DIR, DRIVE_INPUT_DIR

    raise ValueError("INPUT_MODE must be either 'upload' or 'drive'.")


def language_suffix(target_language: str) -> str:
    if OUTPUT_SUFFIX:
        return OUTPUT_SUFFIX.strip().lstrip(".")
    return target_language.strip().lower().split()[0][:2] or "translated"


def resolve_model_name() -> str | None:
    if MODEL_NAME is None:
        return None
    if MODEL_NAME == "custom":
        custom_model = CUSTOM_MODEL_NAME.strip()
        if not custom_model:
            raise ValueError('CUSTOM_MODEL_NAME is required when MODEL_NAME="custom".')
        return custom_model
    return str(MODEL_NAME).strip()


def output_path_for(input_file: Path, output_root: Path, input_root: Path | None = None) -> Path:
    if INPUT_MODE == "drive" and PRESERVE_DRIVE_SUBFOLDERS and input_root is not None:
        try:
            relative_parent = input_file.parent.relative_to(input_root)
            target_dir = output_root / relative_parent
        except ValueError:
            target_dir = output_root
    else:
        target_dir = output_root
    target_dir.mkdir(parents=True, exist_ok=True)
    suffix = language_suffix(TARGET_LANGUAGE)
    return target_dir / f"{input_file.stem}.{suffix}.srt"


def validate_unique_output_paths(input_paths: list[Path], output_root: Path, input_root: Path | None = None) -> None:
    seen = {}
    for input_path in input_paths:
        output_path = output_path_for(input_path, output_root, input_root)
        key = str(output_path.resolve() if output_path.exists() else output_path.absolute())
        if key in seen:
            raise ValueError(
                f"Duplicate output path would be created for {seen[key]} and {input_path}: {output_path}. "
                "Set PRESERVE_DRIVE_SUBFOLDERS=True, change OUTPUT_SUFFIX, or rename one input file."
            )
        seen[key] = input_path


In [ ]:
import uuid

api_key = get_secret("gemini", "GEMINI_API_KEY") or get_secret("gemini1")
api_key2 = (
    get_secret("gemini_backup", "GEMINI_API_KEY2")
    or get_secret("gemini2")
    or get_secret("gemini_backup", "GEMINI_API_KEY_BACKUP")
)

if not api_key:
    raise RuntimeError(
        "Gemini API key is missing. Add a Colab Secret named 'gemini' "
        "or set the GEMINI_API_KEY environment variable."
    )

input_files, output_dir, input_root = collect_input_files()
selected_model_name = resolve_model_name()
validate_unique_output_paths(input_files, output_dir, input_root)
translated_files = []

print(f"Selected {len(input_files)} SRT file(s).")
print(f"Output folder: {output_dir}")
print(f"Target: {TARGET_LANGUAGE}")
print(f"Model: {selected_model_name or 'package default'}")

for input_path in input_files:
    output_file = output_path_for(input_path, output_dir, input_root)
    previous_mtime_ns = output_file.stat().st_mtime_ns if output_file.exists() else None
    if output_file.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"Output already exists: {output_file}. Set OVERWRITE_OUTPUT=True for a fresh run."
        )
    temp_output_file = output_file.with_name(f".{output_file.stem}.{uuid.uuid4().hex}.tmp{output_file.suffix}")

    gst.gemini_api_key = api_key
    gst.gemini_api_key2 = api_key2
    gst.target_language = TARGET_LANGUAGE
    gst.input_file = str(input_path)
    gst.output_file = str(temp_output_file)
    gst.model_name = selected_model_name
    gst.batch_size = BATCH_SIZE
    gst.start_line = START_LINE
    gst.description = DESCRIPTION
    gst.resume = False
    gst.free_quota = FREE_QUOTA
    gst.skip_upgrade = SKIP_UPGRADE_CHECK
    gst.quiet = QUIET

    print()
    print(f"Input:  {input_path}")
    print(f"Output: {output_file}")

    try:
        gst.translate()

        if not temp_output_file.exists() or temp_output_file.stat().st_size == 0:
            raise RuntimeError(f"Translation failed or produced an empty file for {input_path.name}.")

        temp_output_file.replace(output_file)
    finally:
        if temp_output_file.exists():
            temp_output_file.unlink()

    if previous_mtime_ns is not None and output_file.stat().st_mtime_ns == previous_mtime_ns:
        raise RuntimeError(f"Translation did not update the existing output for {input_path.name}.")

    translated_files.append(output_file)
    print(f"Done: {output_file}")

print()
print(f"Translated {len(translated_files)} file(s).")

if INPUT_MODE == "drive":
    print(f"Output files are saved in Google Drive: {output_dir}")
elif AUTO_DOWNLOAD and files is not None:
    for output_file in translated_files:
        files.download(str(output_file))
    print(f"Downloaded {len(translated_files)} file(s).")
else:
    print(f"AUTO_DOWNLOAD=False or not running in Colab. Output files are ready at: {output_dir}")
